# Confluence Page Link Graph
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Graphs · **Difficulty/Frequency:** Common (6/10)


## Concepts

**What this problem is really testing:**
- Directed graphs stored as adjacency lists — both a forward AND a reverse version
- BFS, for finding the shortest path on a graph where every edge counts the same
- Tarjan's algorithm, for finding groups of pages that all link back to each other (strongly connected components)

**Why each one shows up here:**
- Pages linking to pages *is* a directed graph.
- `get_inbound` needing to be fast is exactly why we keep a *reverse* adjacency map alongside the normal (forward) one — a classic "spend more memory to save time" trade.
- `find_path` wanting the *shortest* path on an unweighted graph is the textbook use case for BFS.
- `find_cycles` wanting "groups of pages that can all reach each other" is precisely what an SCC (strongly connected component) decomposition answers.

**The one idea to hold onto:** most of this problem is just "pick the right way to store the graph for each question you'll actually be asked" — forward edges for `get_outbound`, reverse edges for `get_inbound`, BFS for `find_path`, and a linear-time SCC algorithm for `find_cycles`.

---

### Quick primers — the building blocks used below

**Directed graph, stored as an adjacency list.**
- A directed graph is nodes plus directed edges: `A -> B` means "A links to B", not necessarily the other way around.
- An adjacency list stores, for each node, the list of nodes it points to — `dict[node] -> list[node]`.
- **Cost:** O(1) to look up one node's outgoing edges; O(V + E) total space for the whole graph.

**Breadth-First Search (BFS).**
- BFS explores a graph in "waves" by distance from the start: visit the start, then everything one edge away, then everything two edges away, and so on.
- It uses a **queue** (FIFO — first in, first out) so it always expands the *oldest* discovered node next.
- Because it explores strictly in order of distance, **the first time BFS reaches a target, it has already found the shortest path to it** (shortest measured in number of edges, when every edge is equally weighted).
- **Cost:** O(V + E) — every node and edge is looked at at most once.
- **In Python:** use `collections.deque` for the queue — it gives O(1) append/popleft. A plain `list` would make removing from the front O(n), slowing everything down.

**What is a Queue?**
- A queue is FIFO: items come out in the same order they went in — the opposite of a stack (LIFO, last in first out).
- BFS's queue holds "discovered but not yet expanded" nodes. Always processing the earliest-discovered one first is exactly what produces the wave-by-distance search order.

**Strongly Connected Components (SCC) and Tarjan's algorithm.**
- In a directed graph, a strongly connected component is a group of nodes where *every* node can reach *every other* node in that group, following the edges.
- Tarjan's algorithm finds all of these in a **single DFS pass** — no need to run separate searches.
- It tracks, per node: when it was discovered (`indices`), and the earliest-discovered node it can reach that's still "open" (`lowlink`) — plus a stack of currently-open nodes.
- When a node's `lowlink` equals its own discovery index, that node is the "root" of a finished component, and it gets popped off the stack.
- **Cost:** O(V + E) — each node is pushed/popped from the stack exactly once, and each edge is looked at exactly once.


## Problem Statement

Build a directed link graph over Confluence pages, supporting:
- `add_page(page_id, outbound_links)` -- register a page and what it links to.
- `get_outbound(page_id)` / `get_inbound(page_id)` -- pages it links to / pages that link to it.
- `find_path(from_id, to_id)` -- a **shortest** path (list of page IDs), or `None`.
- `orphaned_pages()` -- pages with zero inbound links.

**Constraints:** self-loops are ignored everywhere; a page may be referenced (as a link target) before it's added via `add_page`; up to 100,000 pages.

**Follow-up:** `find_cycles()` -- every strongly connected component of size > 1 (a genuine multi-page cycle).


### `find_path` -- Approach 1 -- Naive (DFS)

**Idea:** a plain depth-first search from `from_id` will find *a* path to `to_id` if one exists -- but DFS has no notion of "shortest"; it commits to one branch and only backtracks when it dead-ends, so the path it happens to find can be far longer than necessary.

**Time complexity:** O(V + E) to find *some* path (same asymptotic cost as BFS!) -- but the path it returns has no length guarantee.

**Why it's not enough:** the problem explicitly asks for a **shortest** path. DFS's traversal order has no relationship to path length, so "first path found" and "shortest path" are unrelated for DFS -- this is a correctness gap, not a speed one.


In [ ]:
from collections import deque
from typing import Dict, List, Optional, Set


def find_path_dfs(out_adj: Dict[str, List[str]], from_id: str, to_id: str) -> Optional[List[str]]:
    """DFS: finds A path, with no guarantee it's the shortest one."""
    if from_id not in out_adj or to_id not in out_adj:
        return None

    visited: Set[str] = {from_id}
    path = [from_id]

    def dfs(node: str) -> bool:
        if node == to_id:
            return True
        for neighbor in out_adj.get(node, []):
            if neighbor == node or neighbor in visited:
                continue
            visited.add(neighbor)
            path.append(neighbor)
            if dfs(neighbor):
                return True
            path.pop()                  # backtrack -- this branch didn't reach to_id
        return False

    return path if dfs(from_id) else None


### `find_path` -- Approach 2 -- Optimal (BFS with a parent map)

**Idea:** BFS from `from_id`, recording each newly-discovered node's parent. The moment `to_id` is discovered, walk the parent chain backward to reconstruct the path, then reverse it. Because BFS discovers nodes in strict order of distance, this is guaranteed shortest.

**Time complexity:** O(V + E) -- every node is enqueued once, every edge examined once.

**Space complexity:** O(V) for the parent map and queue.


In [ ]:
def find_path_bfs(out_adj: Dict[str, List[str]], from_id: str, to_id: str) -> Optional[List[str]]:
    if from_id == to_id:
        return [from_id] if from_id in out_adj else None
    if from_id not in out_adj or to_id not in out_adj:
        return None

    parent: Dict[str, Optional[str]] = {from_id: None}
    queue = deque([from_id])

    while queue:
        current = queue.popleft()
        for neighbor in out_adj.get(current, []):
            if neighbor == current or neighbor in parent:   # self-loop, or already discovered
                continue
            parent[neighbor] = current
            if neighbor == to_id:
                path = []
                node: Optional[str] = to_id
                while node is not None:
                    path.append(node)
                    node = parent[node]
                return path[::-1]
            queue.append(neighbor)

    return None


### The full `PageLinkGraph` class

Maintains `out_adj` (forward edges) and `in_adj` (reverse edges) side by side so both `get_outbound` and `get_inbound` are O(1) lookups, plus `known_pages` (pages explicitly `add_page`'d, vs. pages only seen as a link target so far) and `in_degree` for O(1) `orphaned_pages` filtering. `find_cycles` is Tarjan's algorithm over `out_adj`.

**Complexity summary:** `add_page` is O(k) for k outbound links; `get_outbound`/`get_inbound`/`orphaned_pages` (per call, given the maintained maps) are O(1)/O(1)/O(number of known pages); `find_path` is O(V + E) via BFS; `find_cycles` is O(V + E) via Tarjan's.

**Space:** O(V + E) -- both adjacency maps together store every edge twice (once forward, once reverse).


In [ ]:
class PageLinkGraph:
    def __init__(self) -> None:
        self.out_adj: Dict[str, List[str]] = {}
        self.in_adj: Dict[str, List[str]] = {}
        self.known_pages: Set[str] = set()
        self.in_degree: Dict[str, int] = {}

    def _ensure(self, page_id: str) -> None:
        self.out_adj.setdefault(page_id, [])
        self.in_adj.setdefault(page_id, [])
        self.in_degree.setdefault(page_id, 0)

    def add_page(self, page_id: str, outbound_links: List[str]) -> None:
        self.known_pages.add(page_id)
        self._ensure(page_id)
        for target in outbound_links:
            if target == page_id:
                continue                       # ignore self-loops
            self._ensure(target)               # target may not be add_page'd yet
            self.out_adj[page_id].append(target)
            self.in_adj[target].append(page_id)
            self.in_degree[target] += 1

    def get_outbound(self, page_id: str) -> List[str]:
        return self.out_adj.get(page_id, [])

    def get_inbound(self, page_id: str) -> List[str]:
        return self.in_adj.get(page_id, [])

    def find_path(self, from_id: str, to_id: str) -> Optional[List[str]]:
        return find_path_bfs(self.out_adj, from_id, to_id)

    def orphaned_pages(self) -> List[str]:
        return [pid for pid in self.known_pages if self.in_degree.get(pid, 0) == 0]

    def find_cycles(self) -> List[List[str]]:
        """Tarjan's algorithm: all strongly connected components of size > 1."""
        index_counter = [0]
        stack: List[str] = []
        on_stack: Set[str] = set()
        indices: Dict[str, int] = {}
        lowlink: Dict[str, int] = {}
        sccs: List[List[str]] = []

        def strongconnect(v: str) -> None:
            indices[v] = lowlink[v] = index_counter[0]
            index_counter[0] += 1
            stack.append(v)
            on_stack.add(v)

            for w in self.out_adj.get(v, []):
                if w == v:
                    continue
                if w not in indices:
                    strongconnect(w)
                    lowlink[v] = min(lowlink[v], lowlink[w])
                elif w in on_stack:
                    lowlink[v] = min(lowlink[v], indices[w])

            if lowlink[v] == indices[v]:            # v is the root of a complete SCC
                scc = []
                while True:
                    w = stack.pop()
                    on_stack.remove(w)
                    scc.append(w)
                    if w == v:
                        break
                if len(scc) > 1:
                    sccs.append(scc)

        for pid in self.out_adj:
            if pid not in indices:
                strongconnect(pid)
        return sccs


## Verification

Run the worked example, both `find_path` approaches, and every edge case the Talking Points call out.

In [ ]:
g = PageLinkGraph()
g.add_page("p1", ["p2", "p3"])
g.add_page("p2", ["p3"])
g.add_page("p3", [])

assert g.get_outbound("p1") == ["p2", "p3"]
assert sorted(g.get_inbound("p3")) == ["p1", "p2"]
assert g.find_path("p1", "p3") in (["p1", "p2", "p3"], ["p1", "p3"])
assert len(g.find_path("p1", "p3")) == 2         # BFS guarantees the SHORTEST: ["p1","p3"], length 2
assert g.orphaned_pages() == ["p1"]

# BFS vs DFS: construct a graph where DFS's path is provably longer than BFS's
g2 = PageLinkGraph()
g2.add_page("A", ["B", "D"])
g2.add_page("B", ["C"])
g2.add_page("C", ["D"])
g2.add_page("D", [])
bfs_path = find_path_bfs(g2.out_adj, "A", "D")
assert bfs_path == ["A", "D"]                      # shortest: direct edge
dfs_path = find_path_dfs(g2.out_adj, "A", "D")
assert dfs_path in (["A", "D"], ["A", "B", "C", "D"])   # DFS depends on adjacency order -- not guaranteed shortest

# Self-loops are ignored everywhere
g3 = PageLinkGraph()
g3.add_page("x", ["x", "y"])                        # self-loop to x, real link to y
assert g3.get_outbound("x") == ["y"]                 # self-loop excluded
assert g3.find_path("x", "x") == ["x"]               # trivially reachable (it's a known page)

# Referenced-before-added: "z" appears as a target before add_page("z", ...)
g4 = PageLinkGraph()
g4.add_page("m", ["z"])
assert g4.get_inbound("z") == ["m"]
assert g4.orphaned_pages() == ["m"]                  # "z" not yet known -> not reported as orphaned OR not
assert "z" not in g4.known_pages
g4.add_page("z", [])
assert "z" in g4.known_pages
assert g4.orphaned_pages() == ["m"]                  # z now known but has an inbound link from m -> not orphaned

# No path exists
assert g.find_path("p3", "p1") is None               # p3 has no outbound links at all
g5 = PageLinkGraph()
g5.add_page("a", [])
g5.add_page("b", [])
assert g5.find_path("a", "b") is None                # disconnected

# find_cycles: a genuine 3-cycle plus an unrelated acyclic pair
g6 = PageLinkGraph()
g6.add_page("c1", ["c2"])
g6.add_page("c2", ["c3"])
g6.add_page("c3", ["c1"])          # closes the cycle c1 -> c2 -> c3 -> c1
g6.add_page("solo1", ["solo2"])    # NOT a cycle -- solo2 doesn't link back
g6.add_page("solo2", [])
cycles = g6.find_cycles()
assert len(cycles) == 1
assert set(cycles[0]) == {"c1", "c2", "c3"}

# Self-loop alone must NOT register as a cycle (size-1 SCCs are excluded by design)
g7 = PageLinkGraph()
g7.add_page("only", ["only"])       # self-loop, ignored
assert g7.find_cycles() == []

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **`add_page` called multiple times for the same page -- replace or append?** The current implementation **appends**, so calling it twice would create duplicate edges and double-count `in_degree`. Whether that's desired depends on the product semantics (a page's link *list* being re-saved should probably *replace* old outbound edges, requiring you to first remove the old edges' effects on `in_adj`/`in_degree` before adding the new ones).
- **Weighted links (some links "more important" than others).** Swap BFS for **Dijkstra's algorithm** (a min-heap keyed by cumulative distance) -- it generalizes shortest-path to non-negative weights, at O((V + E) log V) instead of BFS's O(V + E).
- **Graph too large for one machine.** Sharding by page ID is the natural first move, but `find_path` and `find_cycles` then need cross-shard coordination (distributed BFS layer-by-layer with message passing, or graph-parallel frameworks) -- meaningfully harder, worth naming as "a different system," not a tweak.
- **Detecting cycle membership without full SCC decomposition.** Tarjan's already answers this directly -- "is this page in a cycle?" is just "is this page in an SCC of size > 1?", a single lookup once `find_cycles()` has run once.
- **All paths between two pages, not just the shortest.** Exponential in general (a graph can have exponentially many simple paths), so in practice you'd cap path length or count, or only enumerate up to a small hop limit via bounded BFS/DFS.
- **What a cycle *means* for Confluence navigation.** A cycle isn't inherently a bug -- a documentation hub linking out to related pages that link back to the hub is normal and often intentional. What's worth flagging is a **large** SCC: many pages all mutually reachable can signal a maze users can't navigate *out* of, which is a UX smell even though it's not a correctness problem.


## Empirical complexity check

Both `find_path` (BFS) and `find_cycles` (Tarjan's) are O(V + E). We build graphs of growing size with a **fixed average out-degree**, so E scales proportionally with V, and confirm total work scales linearly with the number of pages.

| Growth when n doubles | Implies |
|---|---|
| ~2x | linear in V + E |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

sys.setrecursionlimit(20000)   # Tarjan's recurses once per unvisited node along a DFS branch


def make_chain_graph(n):
    # A long chain p0 -> p1 -> ... -> p(n-1), plus a few extra edges for realistic out-degree.
    # Not a cycle: this specifically times find_path's BFS worst case (search the whole chain).
    g = PageLinkGraph()
    for i in range(n):
        targets = [f"p{i+1}"] if i + 1 < n else []
        if i + 2 < n:
            targets.append(f"p{i+2}")     # small branching factor, still O(V+E) with E = O(V)
        g.add_page(f"p{i}", targets)
    return (g, "p0", f"p{n - 1}")


def run_find_path(g, from_id, to_id):
    g.find_path(from_id, to_id)


def make_cycle_graph(n):
    # One big n-page cycle plus light branching -- exercises find_cycles' full Tarjan pass.
    g = PageLinkGraph()
    for i in range(n):
        targets = [f"p{(i + 1) % n}"]
        if i + 3 < n:
            targets.append(f"p{i + 3}")
        g.add_page(f"p{i}", targets)
    return (g,)


def run_find_cycles(g):
    g.find_cycles()


solutions_path = {"find_path (BFS)": run_find_path}
sizes = [2000, 4000, 8000, 16000]
benchmark(solutions_path, make_chain_graph, sizes, plot=True)

solutions_cycles = {"find_cycles (Tarjan)": run_find_cycles}
benchmark(solutions_cycles, make_cycle_graph, sizes, plot=True)


## 🧩 Patterns Learned

- **Maintain forward AND reverse adjacency when both directions must be fast.** `get_inbound` without a reverse map would be an O(V + E) scan every call; the reverse map trades O(V + E) extra space for O(1) queries -- state this trade-off out loud.
- **BFS = shortest path on unweighted graphs; DFS finds *a* path, not necessarily short.** Whenever "shortest"/"minimum hops"/"fewest steps" appears with unweighted edges, that's BFS, full stop.
- **Track a "known" set separately from "has adjacency-map entries."** Any graph that allows nodes to be referenced before they're formally added needs this distinction, or "orphaned"/"exists" queries silently produce wrong-but-plausible answers.
- **Tarjan's `lowlink == index` is "this node closes off a complete SCC."** The one-DFS-pass elegance comes from tracking, per node, the earliest-discovered node reachable from it that's still "open" (on the stack) -- when that earliest reachable node IS itself, its whole component is done.
- **Related problems:** Course Schedule (cycle detection in a DAG, a degenerate case of "no SCC of size > 1"), Number of Islands (BFS/DFS for connectivity, no shortest-path angle), Network Delay Time (Dijkstra when edges are weighted).
- **Common pitfalls:** forgetting to skip self-loops consistently across every method (easy to remember in `add_page`, easy to forget in `find_cycles`'s `w == v` check); using DFS when "shortest" was asked for; not distinguishing "referenced as a link target" from "actually added" when computing orphaned pages.
